# Exploration des données Olist : qualité, saisonnalité et baisses des ventes

**Objectif :** comprendre les données avant d'écrire les modèles dbt et les tableaux de bord. Le notebook répond à six questions :

1. Les données sont-elles fiables ? (valeurs manquantes, doublons, clés, dates incohérentes)
2. Quelle période peut-on analyser sans se tromper ? (mois incomplets au début et à la fin)
3. Les ventes progressent-elles ? (tendance, croissance d'une année sur l'autre)
4. Y a-t-il une saisonnalité ? (jour de la semaine, heure, mois, Black Friday)
5. Quand les ventes baissent-elles anormalement, et pourquoi ?
6. Quels produits, régions et délais de livraison méritent un tableau de bord ?

**Données :** les CSV Olist téléchargés par `python -m pipeline_retail download-olist` dans `data/olist_source/`. Les enrichissements (jours fériés, États, population) sont facultatifs : s'ils sont absents, les cellules concernées l'indiquent et passent leur tour.

**Comment lire ce notebook :** chaque section se termine par un encadré *À retenir*. Les chiffres viennent des sorties des cellules : ce sont eux qui font foi, pas le texte.

## 0. Préparer les données

Une seule fois, dans le terminal, à la racine du projet :

```bash
python -m pip install -r requirements-dev.txt                 # ajoute statsmodels
python -m pipeline_retail download-olist                      # CSV Olist (déjà fait)
python -m pipeline_retail extract-reference --date 2026-10-27 # facultatif : fériés, États, population
```

Ensuite, dans VS Code, ouvrir ce notebook et choisir le noyau `.venv` en haut à droite.

In [ ]:
# --- Bibliothèques standard ---
import json                      # lire les fichiers bruts NDJSON (une ligne = un objet JSON)
import sys                       # modifier le chemin de recherche des modules Python
from pathlib import Path         # manipuler les chemins de fichiers sans se soucier du système

# --- Bibliothèques d'analyse ---
import matplotlib.pyplot as plt  # graphiques de base
import numpy as np               # calcul numérique (médianes, logarithmes...)
import pandas as pd              # tableaux de données (DataFrame)
import seaborn as sns            # graphiques statistiques plus lisibles (heatmap...)
from statsmodels.tsa.seasonal import STL  # décomposition tendance / saisonnalité / résidu

# Afficher toutes les colonnes et des tableaux larges, plutôt que des "..."
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
# Afficher les nombres décimaux avec 2 chiffres après la virgule
pd.set_option("display.float_format", "{:,.2f}".format)
# Style homogène pour tous les graphiques
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)

# Le notebook fonctionne qu'il soit lancé depuis la racine du projet ou depuis notebooks/ :
# on remonte d'un dossier si le dossier courant s'appelle "notebooks".
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# On ajoute la racine au chemin Python pour pouvoir importer le paquet pipeline_retail.
sys.path.insert(0, str(ROOT))

from pipeline_retail import config  # la liste des fichiers et dossiers est définie à un seul endroit

SOURCE_DIR = ROOT / "data" / "olist_source"  # CSV téléchargés depuis Kaggle
RAW_DIR = ROOT / "data" / "raw"              # fichiers bruts écrits par le pipeline
print("Dossier des CSV :", SOURCE_DIR)
print("CSV présents :", sorted(p.name for p in SOURCE_DIR.glob("*.csv")))

## 1. Charger les tables

Olist est un **schéma en étoile** : une table de faits (les commandes et leurs articles) et des tables de dimensions (clients, produits, vendeurs). Les clés de jointure sont `order_id`, `customer_id`, `product_id` et `seller_id`.

| Table | Une ligne = | Clé |
|---|---|---|
| orders | une commande | `order_id` |
| order_items | un article d'une commande | `order_id` + `order_item_id` |
| order_payments | un moyen de paiement d'une commande | `order_id` + `payment_sequential` |
| order_reviews | un avis client | `review_id` (voir les doublons plus bas) |
| customers | un client *pour une commande* | `customer_id` |
| products | un produit | `product_id` |

In [ ]:
def read_olist(table: str, date_cols: tuple = ()) -> pd.DataFrame:
    # Nom du fichier CSV correspondant à la table (défini dans pipeline_retail/config.py)
    path = SOURCE_DIR / config.OLIST_FILES[table]
    # encoding="utf-8-sig" retire le caractère invisible (BOM) au début de certains fichiers ;
    # parse_dates convertit directement les colonnes de dates en vrais types date/heure.
    return pd.read_csv(path, encoding="utf-8-sig", parse_dates=list(date_cols))


# Colonnes de dates de la table des commandes : toutes les étapes de la vie d'une commande
ORDER_DATES = ("order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
               "order_delivered_customer_date", "order_estimated_delivery_date")

orders = read_olist("orders", ORDER_DATES)
items = read_olist("order_items", ("shipping_limit_date",))
payments = read_olist("order_payments")
reviews = read_olist("order_reviews", ("review_creation_date", "review_answer_timestamp"))
customers = read_olist("customers")
products = read_olist("products")
categories = read_olist("category_translation")

# Un petit dictionnaire pour boucler sur toutes les tables dans la suite
tables = {"orders": orders, "order_items": items, "order_payments": payments,
          "order_reviews": reviews, "customers": customers, "products": products,
          "category_translation": categories}
# Nombre de lignes et de colonnes de chaque table
pd.DataFrame({name: df.shape for name, df in tables.items()}, index=["lignes", "colonnes"]).T

In [ ]:
# Aperçu de 3 commandes : à quoi ressemble une ligne ?
orders.head(3)

## 2. Qualité des données

On vérifie quatre choses, dans cet ordre : les valeurs manquantes, l'unicité des clés, l'intégrité des jointures, puis la cohérence des dates et des montants.

In [ ]:
# Taux de valeurs manquantes (en %) pour chaque colonne qui en contient
missing = []
for name, df in tables.items():
    rate = df.isna().mean() * 100          # proportion de cases vides par colonne
    for col, pct in rate[rate > 0].items():  # on ne garde que les colonnes concernées
        missing.append({"table": name, "colonne": col, "% manquant": pct})
pd.DataFrame(missing).sort_values("% manquant", ascending=False)

**Lecture attendue :** les dates de livraison manquent surtout pour les commandes non livrées (annulées, en cours), ce qui est normal. Les titres et messages d'avis sont facultatifs pour le client : leur absence n'est pas une erreur. Une catégorie produit manquante, en revanche, devra être remplacée par une valeur explicite (par exemple « inconnue ») dans dbt.

In [ ]:
# Unicité des clés : une clé dupliquée fausserait les comptages après jointure.
keys = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
}
checks = []
for name, cols in keys.items():
    df = tables[name]
    # duplicated() marque chaque ligne dont la combinaison de clés a déjà été vue
    checks.append({"table": name, "clé": " + ".join(cols),
                   "doublons": int(df.duplicated(subset=cols).sum())})
pd.DataFrame(checks)

In [ ]:
# Les avis : une commande peut-elle avoir plusieurs avis, et un avis plusieurs commandes ?
reviews_per_order = reviews.groupby("order_id")["review_id"].nunique()  # avis distincts par commande
orders_per_review = reviews.groupby("review_id")["order_id"].nunique()  # commandes distinctes par avis
print("Commandes avec plusieurs avis :", int((reviews_per_order > 1).sum()))
print("Avis rattachés à plusieurs commandes :", int((orders_per_review > 1).sum()))
print("Lignes d'avis entièrement dupliquées :", int(reviews.duplicated().sum()))

**Décision pour dbt :** pour obtenir *une note par commande*, on garde l'avis le plus récent de chaque commande (`review_answer_timestamp` le plus tardif). Sans cette règle, les commandes avec deux avis compteraient double dans la note moyenne.

In [ ]:
# Intégrité des jointures : chaque article doit pointer vers une commande et un produit existants.
def orphans(child: pd.DataFrame, parent: pd.DataFrame, key: str) -> int:
    # isin() vérifie si chaque valeur de la clé existe dans la table parente ;
    # le ~ inverse le résultat : on compte les clés SANS correspondance.
    return int((~child[key].isin(parent[key])).sum())

print("Articles sans commande :", orphans(items, orders, "order_id"))
print("Articles sans produit :", orphans(items, products, "product_id"))
print("Paiements sans commande :", orphans(payments, orders, "order_id"))
print("Avis sans commande :", orphans(reviews, orders, "order_id"))
print("Commandes sans client :", orphans(orders, customers, "customer_id"))
# Une commande sans article ne génère aucun chiffre d'affaires : combien y en a-t-il, et quels statuts ?
no_items = orders[~orders["order_id"].isin(items["order_id"])]
print("Commandes sans article :", len(no_items))
no_items["order_status"].value_counts()

In [ ]:
# Statuts des commandes : lesquels doivent compter dans le chiffre d'affaires ?
status = orders["order_status"].value_counts().to_frame("commandes")
status["%"] = status["commandes"] / len(orders) * 100
status

**Décision pour dbt :** le chiffre d'affaires exclut les statuts `canceled` et `unavailable` (commandes qui n'ont pas abouti). Les autres statuts (`delivered`, `shipped`, `invoiced`...) sont des ventes réalisées à des stades différents. Les tableaux de bord pourront afficher le taux d'annulation à part, comme indicateur de qualité.

In [ ]:
# Cohérence des dates : une étape ne peut pas précéder la précédente.
o = orders
date_checks = {
    "validation avant l'achat": (o["order_approved_at"] < o["order_purchase_timestamp"]).sum(),
    "remise au transporteur avant validation": (o["order_delivered_carrier_date"] < o["order_approved_at"]).sum(),
    "livraison client avant remise au transporteur": (o["order_delivered_customer_date"] < o["order_delivered_carrier_date"]).sum(),
    "livrée mais sans date de livraison": ((o["order_status"] == "delivered") & o["order_delivered_customer_date"].isna()).sum(),
}
# Les comparaisons avec une date manquante donnent False : elles ne sont donc pas comptées.
pd.Series(date_checks, name="commandes").to_frame()

In [ ]:
# Cohérence des montants : ce que le client paie doit correspondre aux articles + frais de port.
# Somme des prix et des frais de port par commande
order_amounts = items.groupby("order_id")[["price", "freight_value"]].sum()
order_amounts["items_total"] = order_amounts["price"] + order_amounts["freight_value"]
# Somme des paiements par commande (une commande peut être payée en plusieurs fois ou par plusieurs moyens)
paid = payments.groupby("order_id")["payment_value"].sum().rename("paid_total")
# join() aligne les deux tableaux sur order_id
compare = order_amounts.join(paid, how="inner")
compare["écart"] = compare["paid_total"] - compare["items_total"]
# Une tolérance d'un centime absorbe les arrondis
gap = compare[compare["écart"].abs() > 0.01]
print(f"Commandes comparées : {len(compare):,}")
print(f"Commandes avec un écart > 0,01 BRL : {len(gap):,} ({len(gap) / len(compare):.1%})")
gap["écart"].describe()

**Lecture :** de petits écarts positifs s'expliquent souvent par les intérêts des paiements en plusieurs fois (`payment_installments`). Ils ne remettent pas en cause l'analyse, mais il faut **choisir une définition unique du chiffre d'affaires** et s'y tenir dans tous les tableaux de bord.

**Décision pour dbt :** chiffre d'affaires = somme de `price` (ventes de marchandises, hors frais de port). Les frais de port (`freight_value`) sont suivis à part.

> **À retenir (qualité)**
> - Les clés principales sont uniques, sauf les avis : une règle de dédoublonnage est nécessaire.
> - Les statuts `canceled` et `unavailable` sont exclus du chiffre d'affaires.
> - Le chiffre d'affaires est calculé à partir des articles (`price`), pas des paiements.
> - Ces règles deviennent des **tests dbt** (`unique`, `not_null`, `relationships`) à l'étape 4.

## 3. Construire la table de travail

On assemble une ligne par commande, avec sa date, son État, son chiffre d'affaires et son statut. C'est l'équivalent de la future table dbt `fct_orders`.

In [ ]:
# Chiffre d'affaires et nombre d'articles par commande
per_order = items.groupby("order_id").agg(
    revenue=("price", "sum"),          # chiffre d'affaires marchandises
    freight=("freight_value", "sum"),  # frais de port
    n_items=("order_item_id", "count"),  # nombre d'articles
)

fct = (
    orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp"]]
    .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
    .merge(per_order, on="order_id", how="left")  # left : on garde aussi les commandes sans article
)
# Une commande sans article a un chiffre d'affaires nul, pas inconnu
fct[["revenue", "freight", "n_items"]] = fct[["revenue", "freight", "n_items"]].fillna(0)
# Une commande est "valide" si elle n'a été ni annulée ni déclarée indisponible
fct["is_valid"] = ~fct["order_status"].isin(["canceled", "unavailable"])
# Colonnes de temps utiles pour la saisonnalité
ts = fct["order_purchase_timestamp"]
fct["day"] = ts.dt.normalize()                 # la date sans l'heure
fct["month"] = ts.dt.to_period("M")            # le mois (2017-11)
fct["weekday"] = ts.dt.dayofweek               # 0 = lundi ... 6 = dimanche
fct["hour"] = ts.dt.hour                       # heure d'achat (heure de Brasília)
fct.head()

### Lien avec le pipeline : le rejeu donne-t-il le même résultat ?

Le fichier brut écrit par `extract-daily` pour le 24/11/2017 doit contenir exactement les commandes de ce jour dans le CSV. Si le fichier n'existe pas, lancer `python -m pipeline_retail extract-daily --date 2017-11-24`.

In [ ]:
raw_file = RAW_DIR / "olist" / "orders" / "date=2017-11-24" / "data.ndjson"
if raw_file.exists():
    # Chaque ligne du fichier brut est un objet JSON : {"run_date", "extracted_at", "source", "record"}
    with open(raw_file, encoding="utf-8") as f:
        raw_rows = [json.loads(line) for line in f if line.strip()]
    print("Exemple d'enveloppe brute :", {k: v for k, v in raw_rows[0].items() if k != "record"})
    # Nombre de commandes du 24/11/2017 dans le CSV source
    expected = int((fct["day"] == "2017-11-24").sum())
    print(f"Fichier brut : {len(raw_rows)} commandes | CSV source : {expected} commandes")
    print("Rejeu cohérent :", len(raw_rows) == expected)
else:
    print("Fichier brut absent : lancer extract-daily --date 2017-11-24 pour faire ce contrôle.")

## 4. Quelle période analyser ?

Olist couvre de septembre 2016 à octobre 2018, mais le début et la fin sont **clairsemés** : la plateforme démarrait, et l'extraction a été arrêtée en cours de mois. Comparer ces mois aux autres ferait apparaître une fausse croissance au début et une **fausse chute** à la fin. C'est le premier piège à éviter.

In [ ]:
# Nombre de commandes et chiffre d'affaires par mois (commandes valides uniquement)
valid = fct[fct["is_valid"]]
monthly = valid.groupby("month").agg(orders=("order_id", "count"), revenue=("revenue", "sum"))
# Nombre de jours distincts avec au moins une commande dans le mois
monthly["active_days"] = valid.groupby("month")["day"].nunique()
monthly["days_in_month"] = monthly.index.days_in_month
monthly

In [ ]:
# Règle explicite pour repérer les mois incomplets : un mois est "complet" si
#   1) il a des commandes au moins 90 % des jours du mois,
#   2) son volume dépasse 20 % du volume mensuel médian.
# Les seuils sont des choix d'analyste : on les affiche pour pouvoir les discuter.
median_orders = monthly["orders"].median()
monthly["complete"] = ((monthly["active_days"] >= 0.9 * monthly["days_in_month"])
                       & (monthly["orders"] >= 0.2 * median_orders))

complete_months = monthly.index[monthly["complete"]]
# La fenêtre d'analyse va du premier au dernier mois complet
START = complete_months.min().start_time       # premier jour du premier mois complet
END = complete_months.max().end_time.normalize()  # dernier jour du dernier mois complet
print("Mois écartés :", [str(m) for m in monthly.index[~monthly["complete"]]])
print(f"Fenêtre d'analyse : du {START:%d/%m/%Y} au {END:%d/%m/%Y}")

In [ ]:
# Visualiser le piège : les mois écartés sont en gris
fig, ax = plt.subplots()
colors = np.where(monthly["complete"], "tab:blue", "lightgray")  # une couleur par barre
ax.bar(monthly.index.astype(str), monthly["orders"], color=colors)
ax.set_title("Commandes valides par mois (en gris : mois incomplets, exclus de l'analyse)")
ax.set_ylabel("commandes")
ax.tick_params(axis="x", rotation=90)  # étiquettes verticales pour qu'elles ne se chevauchent pas
plt.show()

> **À retenir (période)**
> - L'analyse porte sur la fenêtre affichée ci-dessus, pas sur tout le fichier.
> - Dans les tableaux de bord, la même règle doit s'appliquer : sinon, le dernier mois affichera une baisse qui n'existe pas.
> - Conséquence : on dispose d'**une seule année complète** (2017) et d'environ huit mois de 2018. La saisonnalité *annuelle* ne peut pas être établie avec certitude (il faudrait plusieurs années).

## 5. Tendance : les ventes progressent-elles ?

In [ ]:
# Série quotidienne sur la fenêtre d'analyse
daily = (valid[(valid["day"] >= START) & (valid["day"] <= END)]
         .groupby("day").agg(orders=("order_id", "count"), revenue=("revenue", "sum")))
# reindex ajoute les jours sans commande (avec 0) : une série temporelle doit être continue
all_days = pd.date_range(START, END, freq="D")
daily = daily.reindex(all_days, fill_value=0)
daily.index.name = "day"
print("Jours sans aucune commande dans la fenêtre :", int((daily["orders"] == 0).sum()))

# Moyenne mobile sur 28 jours (4 semaines entières) : elle lisse l'effet jour de la semaine
daily["revenue_ma28"] = daily["revenue"].rolling(28, center=True).mean()

fig, ax = plt.subplots()
ax.plot(daily.index, daily["revenue"], color="lightgray", label="chiffre d'affaires du jour")
ax.plot(daily.index, daily["revenue_ma28"], color="tab:blue", label="moyenne mobile 28 jours")
ax.set_title("Chiffre d'affaires quotidien (BRL)")
ax.legend()
plt.show()

In [ ]:
# Croissance d'une année sur l'autre, sur les mêmes mois uniquement (comparaison juste)
m = monthly[monthly["complete"]].copy()
m["year"] = m.index.year
m["month_num"] = m.index.month
# pivot : une ligne par mois de l'année, une colonne par année
pivot = m.pivot_table(index="month_num", columns="year", values="revenue")
years = sorted(pivot.columns)
if len(years) >= 2:
    a, b = years[-2], years[-1]
    common = pivot[[a, b]].dropna()  # les mois présents les deux années
    growth = common[b].sum() / common[a].sum() - 1
    print(f"Mois comparés : {list(common.index)}")
    print(f"Croissance {b} vs {a} sur ces mois : {growth:+.1%}")
pivot

> **À retenir (tendance)** : comparer des périodes identiques (mêmes mois) évite de confondre croissance et saisonnalité. C'est l'indicateur « vs N-1 » à mettre dans le tableau de bord.

## 6. Saisonnalité : quand achète-t-on ?

In [ ]:
in_window = valid[(valid["day"] >= START) & (valid["day"] <= END)]
DAYS_FR = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]

# Nombre moyen de commandes par jour de la semaine.
# On divise par le nombre de lundis, mardis... de la fenêtre : sinon un jour plus fréquent serait avantagé.
n_weekdays = pd.Series(all_days.dayofweek).value_counts().sort_index()
by_weekday = in_window.groupby("weekday")["order_id"].count() / n_weekdays
by_weekday.index = DAYS_FR

# Commandes par jour de la semaine et par heure : une "carte de chaleur"
heat = in_window.pivot_table(index="weekday", columns="hour", values="order_id", aggfunc="count")
heat.index = DAYS_FR

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 3]})
by_weekday.plot.bar(ax=axes[0], color="tab:blue")
axes[0].set_title("Commandes moyennes par jour")
sns.heatmap(heat, ax=axes[1], cmap="Blues", cbar_kws={"label": "commandes"})
axes[1].set_title("Commandes par jour et par heure d'achat")
axes[1].set_xlabel("heure")
plt.tight_layout()
plt.show()

### Décomposition STL : séparer tendance, saisonnalité hebdomadaire et accidents

STL découpe la série en trois parties : **tendance** (évolution de fond), **saisonnalité** (motif qui se répète chaque semaine, `period=7`) et **résidu** (ce qui reste : événements, accidents). L'option `robust=True` empêche les pics comme le Black Friday de déformer la tendance.

On travaille sur le logarithme du chiffre d'affaires : les effets deviennent multiplicatifs (« le lundi fait +10 % ») plutôt qu'additifs, ce qui est plus réaliste pour une activité en croissance.

In [ ]:
# log1p(x) = log(1 + x) : fonctionne aussi pour les jours à 0
series = np.log1p(daily["revenue"])
stl = STL(series, period=7, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(12, 8)
plt.show()

# STL est peu fiable aux bords de la série (démarrage de la plateforme, fin des données) :
# pour les statistiques, on ignore les 4 premières et les 4 dernières semaines.
CORE = (series.index >= START + pd.Timedelta(days=28)) & (series.index <= END - pd.Timedelta(days=28))

# Effet typique de chaque jour de la semaine, en % par rapport à la tendance.
# La médiane résiste mieux que la moyenne aux semaines atypiques.
weekly_effect = (np.expm1(stl.seasonal[CORE]).groupby(series.index[CORE].dayofweek).median() * 100)
weekly_effect.index = DAYS_FR
print("Effet jour de la semaine (% vs tendance) :")
print(weekly_effect.round(1).to_string())

### Les événements : Black Friday et jours fériés

Le Black Friday n'est pas un jour férié : on le calcule (le vendredi qui suit le 4e jeudi de novembre). Les jours fériés viennent du fichier brut Nager.Date, s'il a été extrait.

In [ ]:
def black_friday(year: int) -> pd.Timestamp:
    # Le 4e jeudi de novembre (Thanksgiving aux États-Unis), plus un jour
    first_nov = pd.Timestamp(year=year, month=11, day=1)
    first_thursday = first_nov + pd.Timedelta(days=(3 - first_nov.dayofweek) % 7)  # jeudi = 3
    return first_thursday + pd.Timedelta(weeks=3, days=1)

events = pd.DataFrame({"date": [black_friday(y) for y in config.OLIST_YEARS], "event": "Black Friday"})

# Le fichier des jours fériés le plus récent (dossier date=... le plus récent)
holiday_files = sorted(RAW_DIR.glob("nager/holidays/date=*/data.ndjson"))
if holiday_files:
    with open(holiday_files[-1], encoding="utf-8") as f:
        # On ne garde que le contenu "record" de chaque ligne (la réponse de l'API)
        recs = [json.loads(line)["record"] for line in f if line.strip()]
    hol = pd.DataFrame(recs)
    # Nager.Date liste aussi des fériés régionaux : on garde les fériés nationaux (global = True)
    hol = hol[hol["global"]]
    events = pd.concat([events, pd.DataFrame({"date": pd.to_datetime(hol["date"]), "event": hol["localName"]})])
else:
    print("Jours fériés absents : lancer extract-reference pour les ajouter.")

events = events[(events["date"] >= START) & (events["date"] <= END)].sort_values("date")
# Pour chaque événement : chiffre d'affaires du jour et écart à la tendance (résidu STL, en %)
events["revenue"] = events["date"].map(daily["revenue"])
events["écart vs attendu (%)"] = events["date"].map(np.expm1(stl.resid) * 100)
events

> **À retenir (saisonnalité)**
> - La saisonnalité **hebdomadaire** et **horaire** est mesurable de façon fiable : elle se répète chaque semaine, on a des dizaines de répétitions.
> - Le **Black Friday** est l'événement le plus visible. Il doit apparaître comme annotation dans les tableaux de bord.
> - Les effets des jours fériés se lisent dans la colonne d'écart : un férié ne fait pas forcément baisser les ventes en ligne.
> - La saisonnalité **annuelle** reste une hypothèse : une seule année complète ne permet pas de distinguer « novembre est toujours fort » de « novembre 2017 était fort ».

## 7. Détecter les baisses anormales

Une baisse n'est intéressante que si elle est **inattendue**. Comme pour l'effet hebdomadaire, on ignore les 4 premières et les 4 dernières semaines de la fenêtre, où STL est peu fiable. Le résidu STL mesure justement l'écart à ce qui était attendu (tendance + jour de la semaine). On signale les jours dont le résidu est très bas.

Pour fixer le seuil, on utilise l'écart absolu médian (MAD) plutôt que l'écart-type : il n'est pas gonflé par les pics extrêmes comme le Black Friday.

In [ ]:
resid = stl.resid[CORE]   # résidus hors des bords de la série
# MAD = médiane des écarts à la médiane ; 1,4826 la rend comparable à un écart-type (loi normale)
mad = 1.4826 * np.median(np.abs(resid - np.median(resid)))
z = (resid - np.median(resid)) / mad   # score robuste : combien d'"écarts-types" sous la normale

anomalies = pd.DataFrame({
    "revenue": daily["revenue"],
    "attendu": np.expm1(stl.trend + stl.seasonal),  # niveau attendu sans l'accident
    "écart (%)": np.expm1(resid) * 100,
    "score": z,
}).dropna(subset=["score"])  # les jours hors du cœur de série n'ont pas de score
anomalies["jour"] = [DAYS_FR[d] for d in anomalies.index.dayofweek]  # 0 -> "lun", 6 -> "dim"
# Seuil de -3 : choix prudent, pour ne signaler que des baisses marquées
drops = anomalies[anomalies["score"] < -3].sort_values("score")
peaks = anomalies[anomalies["score"] > 3].sort_values("score", ascending=False)
print(f"Seuil : score < -3 ; {len(drops)} jours de baisse anormale, {len(peaks)} jours de pic")
drops.head(15)

In [ ]:
# Les pics, pour comparaison (le Black Friday devrait être en tête)
peaks.head(10)

In [ ]:
# Vue hebdomadaire : semaines dont le chiffre d'affaires baisse de plus de 15 %
# par rapport à la médiane des 4 semaines précédentes (règle simple, facile à expliquer en réunion).
# La médiane évite qu'une semaine exceptionnelle (Black Friday) fasse paraître les suivantes en baisse.
weekly = daily["revenue"].resample("W-SUN").sum()   # semaines du lundi au dimanche
# On écarte les semaines incomplètes aux bords de la fenêtre
weekly = weekly[(weekly.index - pd.Timedelta(days=6) >= START) & (weekly.index <= END)]
baseline = weekly.shift(1).rolling(4).median()      # médiane des 4 semaines AVANT (shift évite la fuite)
weekly_change = (weekly / baseline - 1) * 100
weekly_drops = weekly_change[weekly_change < -15].round(1)
print("Semaines en baisse de plus de 15 % vs les 4 précédentes (date = dimanche qui clôt la semaine) :")
print(weekly_drops.to_string() if len(weekly_drops) else "aucune")

fig, ax = plt.subplots()
ax.plot(weekly.index, weekly.values, marker="o", markersize=3, label="chiffre d'affaires hebdomadaire")
ax.scatter(weekly_drops.index, weekly[weekly_drops.index], color="tab:red", zorder=3, label="baisse > 15 %")
ax.set_title("Chiffre d'affaires hebdomadaire et semaines en baisse")
ax.legend()
plt.show()

### Fin des données : vraie baisse ou artefact ?

Une baisse sur la toute dernière semaine de la fenêtre est suspecte : c'est souvent le signe que la collecte s'arrête progressivement. On regarde les commandes jour par jour sur les trois dernières semaines.

In [ ]:
# Commandes et chiffre d'affaires des 21 derniers jours de la fenêtre
tail = daily[["orders", "revenue"]].tail(21).copy()
tail["jour"] = [DAYS_FR[d] for d in tail.index.dayofweek]
# Comparaison avec la médiane des 8 semaines précédentes, même jour de la semaine
ref = daily["orders"].iloc[-77:-21]                    # 56 jours avant ces 3 semaines
ref_by_day = ref.groupby(ref.index.dayofweek).median()
tail["habituel (même jour)"] = [ref_by_day[d] for d in tail.index.dayofweek]
tail["écart (%)"] = (tail["orders"] / tail["habituel (même jour)"] - 1) * 100
tail

**Lecture :** si les derniers jours s'effondrent d'un coup, sans retour, il s'agit d'une fin de collecte et non d'une chute des ventes. Dans ce cas, avancer `END` à la veille de l'effondrement (cellule de la section 4) et relancer le notebook.

**Comment interpréter une baisse détectée (avant de conclure) :**

1. **Artefact de données ?** Jours sans aucune commande, fin de période, panne de collecte. Une baisse à 0 commande est presque toujours un problème de données, pas de ventes.
2. **Effet calendrier ?** Lendemain de Black Friday, Carnaval, Noël, jour férié. Croiser avec le tableau des événements.
3. **Effet réel ?** Si ni 1 ni 2 : regarder quelles catégories et quels États ont baissé (section 8). La baisse est-elle générale ou concentrée ?

**Piste à vérifier sur les vraies données :** fin mai 2018, une grève des camionneurs a paralysé les livraisons au Brésil. Si des baisses apparaissent à ces dates, c'est une explication plausible, à confirmer en regardant les délais de livraison des mêmes jours.

Ces méthodes repèrent des **associations dans le temps**, pas des causes. Une baisse qui coïncide avec un férié n'est pas forcément *causée* par ce férié.

## 8. Produits, régions et livraison

In [ ]:
# Chiffre d'affaires par catégorie, avec le nom anglais (plus lisible que le portugais)
cat = (items.merge(orders[["order_id", "order_status", "order_purchase_timestamp"]], on="order_id")
       .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
       .merge(categories, on="product_category_name", how="left"))
cat = cat[~cat["order_status"].isin(["canceled", "unavailable"])
          & cat["order_purchase_timestamp"].between(START, END + pd.Timedelta(days=1))]
# Catégorie absente ou non traduite : valeur explicite plutôt que case vide
cat["category"] = (cat["product_category_name_english"]
                   .fillna(cat["product_category_name"]).fillna("unknown"))
top_cat = cat.groupby("category")["price"].sum().sort_values(ascending=False)
share = top_cat / top_cat.sum() * 100
print(f"Les 10 premières catégories font {share.head(10).sum():.1f} % du chiffre d'affaires "
      f"({len(top_cat)} catégories au total).")
top_cat.head(10).sort_values().plot.barh(title="Top 10 des catégories (chiffre d'affaires, BRL)")
plt.show()

In [ ]:
# Chiffre d'affaires par État du client
by_state = (in_window.groupby("customer_state")
            .agg(orders=("order_id", "count"), revenue=("revenue", "sum"))
            .sort_values("revenue", ascending=False))
by_state["% CA"] = by_state["revenue"] / by_state["revenue"].sum() * 100

# Enrichissement IBGE (facultatif) : chiffre d'affaires par habitant
pop_files = sorted(RAW_DIR.glob("ibge/population/date=*/data.ndjson"))
state_files = sorted(RAW_DIR.glob("ibge/states/date=*/data.ndjson"))
if pop_files and state_files:
    def read_records(path):
        with open(path, encoding="utf-8") as f:
            return [json.loads(line)["record"] for line in f if line.strip()]
    # États : code IBGE (id) -> sigle (SP, RJ...), le sigle étant la clé utilisée par Olist
    states = pd.DataFrame(read_records(state_files[-1]))[["id", "sigla"]]
    states["id"] = states["id"].astype(str)
    # Population : D1C = code de l'État, D3C = année, V = valeur (format SIDRA)
    pop = pd.DataFrame(read_records(pop_files[-1]))[["D1C", "D3C", "V"]]
    pop = pop[pop["D3C"] == "2017"]  # année complète de la fenêtre
    pop["population"] = pd.to_numeric(pop["V"], errors="coerce")  # "..." ou "-" deviennent NaN
    pop = pop.merge(states, left_on="D1C", right_on="id")
    by_state = by_state.join(pop.set_index("sigla")["population"])
    by_state["CA / 1000 hab."] = by_state["revenue"] / by_state["population"] * 1000
else:
    print("Population IBGE absente : lancer extract-reference pour calculer le CA par habitant.")
by_state.head(10)

**Pourquoi le CA par habitant ?** São Paulo domine en valeur absolue simplement parce que c'est l'État le plus peuplé. Rapporter à la population montre où Olist est *proportionnellement* le mieux implanté : c'est la carte utile pour un tableau de bord commercial.

In [ ]:
# Livraison et satisfaction : les commandes en retard ont-elles de moins bonnes notes ?
delivered = orders[orders["order_status"] == "delivered"].dropna(subset=["order_delivered_customer_date"]).copy()
# Délai réel en jours entre l'achat et la livraison
delivered["delay_days"] = (delivered["order_delivered_customer_date"]
                           - delivered["order_purchase_timestamp"]).dt.total_seconds() / 86400
# En retard si livrée après la date estimée annoncée au client
delivered["late"] = delivered["order_delivered_customer_date"].dt.normalize() > delivered["order_estimated_delivery_date"]

# Un avis par commande : le plus récent (règle décidée en section 2)
last_review = (reviews.sort_values("review_answer_timestamp")
               .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
delivered = delivered.merge(last_review, on="order_id", how="left")

print(f"Délai médian de livraison : {delivered['delay_days'].median():.1f} jours")
print(f"Part des commandes en retard : {delivered['late'].mean():.1%}")
delivered.groupby("late")["review_score"].agg(["mean", "count"]).rename(
    index={False: "à l'heure", True: "en retard"})

**Attention, corrélation et non causalité :** si les commandes en retard ont une note plus basse, c'est cohérent avec l'idée que le retard déçoit, mais d'autres facteurs peuvent jouer (catégorie, distance, vendeur). Il faudrait comparer à catégorie et région égales pour aller plus loin.

**Limite du rejeu :** les dates de livraison et les avis sont l'état *final* de chaque commande. Dans un vrai flux quotidien, ces informations arriveraient plusieurs jours après l'achat. Dans dbt, on les traitera comme des attributs de la commande, et non comme des événements datés du jour d'achat.

## 9. Synthèse et décisions pour la suite

Le tableau ci-dessous rassemble les chiffres clés calculés plus haut, pour la documentation du portfolio.

In [ ]:
summary = {
    "Commandes (tout le fichier)": f"{len(orders):,}",
    "Fenêtre d'analyse": f"{START:%d/%m/%Y} → {END:%d/%m/%Y}",
    "Commandes valides dans la fenêtre": f"{len(in_window):,}",
    "Chiffre d'affaires dans la fenêtre (BRL)": f"{in_window['revenue'].sum():,.0f}",
    "Taux d'annulation / indisponibilité": f"{(~fct['is_valid']).mean():.1%}",
    "Jour le plus fort (effet hebdo.)": f"{weekly_effect.idxmax()} ({weekly_effect.max():+.1f} %)",
    "Jour le plus faible (effet hebdo.)": f"{weekly_effect.idxmin()} ({weekly_effect.min():+.1f} %)",
    "Jours de baisse anormale (score < -3)": len(drops),
    "Part des livraisons en retard": f"{delivered['late'].mean():.1%}",
}
pd.Series(summary, name="valeur").to_frame()

### Décisions qui alimentent les modèles dbt (étape 4)

| Décision | Modèle dbt concerné |
|---|---|
| Chiffre d'affaires = somme de `price` ; frais de port à part | `fct_orders`, `fct_order_items` |
| Exclure `canceled` et `unavailable` du chiffre d'affaires, garder un indicateur d'annulation | `fct_orders` |
| Un avis par commande : le plus récent | `stg_olist__order_reviews` |
| Catégorie absente → `unknown`, traduction anglaise via la table de correspondance | `dim_products` |
| Fenêtre d'analyse = mois complets uniquement (règle de la section 4) | `dim_date` (indicateur `is_complete_month`) |
| Série quotidienne continue (jours sans commande = 0) | `fct_daily_sales` |
| Black Friday calculé, jours fériés nationaux de Nager.Date | `dim_date` |
| Population IBGE pour les indicateurs par habitant | `dim_states` |
| Tests : `unique` et `not_null` sur les clés, `relationships` entre faits et dimensions | `schema.yml` |

### Tableaux de bord envisagés (étape 5)

1. **Vue d'ensemble** : chiffre d'affaires, commandes, panier moyen, évolution vs N-1, tendance 28 jours.
2. **Saisonnalité** : carte de chaleur jour × heure, effet jour de la semaine, annotations des événements.
3. **Alertes** : jours et semaines en baisse anormale, avec la catégorie et l'État les plus touchés.
4. **Géographie** : carte par État, en valeur et par habitant.
5. **Expérience client** : délais, retards, note moyenne.

### Limites à mentionner dans la présentation

- Une seule année complète : la saisonnalité annuelle n'est pas démontrée.
- Données historiques rejouées : les délais et les avis reflètent l'état final, pas l'état connu au jour J.
- Les montants sont en réais (BRL) ; la conversion en euros utilisera le taux du jour d'achat (Frankfurter).